In [6]:
import dash
from dash import dcc, html, callback, Output, Input
import dash_bootstrap_components as dbc
import plotly.graph_objects as go
import plotly.express as px
import numpy as np
from scipy.interpolate import make_smoothing_spline
import pandas as pd
import bz2
import _pickle as cPickle

from pathlib import Path


In [7]:
repo_folder = Path('..')
data_folder  = repo_folder / 'data'/ 'this_project' / '6_transporterKO'
metabolomics_folder = data_folder / 'D_big_screen'


In [8]:
median_fn = metabolomics_folder / 'G_median_z_scores_TIC_norm_keio.csv'
df = pd.read_csv(median_fn)
# df_data.rename(columns={'Z-score median': 'Median Z-score'}, inplace=True)

ionMz_annotation_fn =  metabolomics_folder / 'H_ionMz_annotation.csv'
df_ionMz = pd.read_csv(ionMz_annotation_fn)

sample_metadata_fn = metabolomics_folder /  'I_sample_metadata_keio.csv'
df_sample_metadata = pd.read_csv(sample_metadata_fn, index_col=0)

In [9]:

def load_compressed_pickle(filename):
    data = bz2.BZ2File(filename + '.pbz2', 'rb')
    data = cPickle.load(data)
    return data
def save_compressed_pickle(obj, filename):
    with bz2.BZ2File(filename + '.pbz2', 'w') as f:
        cPickle.dump(obj, f)


In [10]:
all_strains = load_compressed_pickle("visualization/data/keio_non_wt_strains_list")


In [11]:
## load mz_
ionMz_annotation_fn = 'visualization/data/H_ionMz_annotation.csv'
df_ionMz = pd.read_csv(ionMz_annotation_fn)
df_ionMz.drop_duplicates(subset=['ionMz'], inplace=True)

In [12]:
# Load sample metadata
sample_metadata_fn = 'visualization/data/I_sample_metadata_keio.csv'
sample_metadata_df = pd.read_csv(sample_metadata_fn)

In [13]:
df2 = pd.merge(df, sample_metadata_df, on=('Batch-Tube', 'Timepoint'), how='left')
df2 = pd.merge(df2, df_ionMz, on='ionMz', how='left')

In [14]:
def avoid_duplicates(arr, idxs = [0], noise_level=1e-3):
    for xi in idxs:
        vals = arr[xi]
        duplicates = pd.Series(vals).duplicated(keep=False)
        if duplicates.any():
            noise = np.random.normal(0, 1e-3, size=duplicates.sum())
            vals[duplicates] += noise
            arr[xi] = vals
    return arr

In [15]:
sample_od_mean = sample_metadata_df.groupby(['Strain','Hours','Batch']).agg({'OD':['mean','std']}).reset_index()
sample_od_mean.columns = ['Strain','Hours','Batch','OD_mean','OD_std']

In [16]:
new_data_dict = {}
for strain in all_strains:
    strain_data = df2.loc[df2['Strain'] == strain]
    batch = strain_data['Batch'].values[0]
    wt_data = df2.loc[(df2['Strain'] == 'WT') & (df2['Batch'] == batch)]
    strain_metabolites = strain_data['Metabolite'].unique()
    strain_dict = {}
    for m in strain_metabolites:
        strain_m_data = strain_data.loc[strain_data['Metabolite'] == m]
        s_arr = strain_m_data[['Hours', 'Z-score median', 'AUC OD', 'Z-score std']].sort_values(by='AUC OD').to_numpy().T
        s_arr = avoid_duplicates(s_arr, idxs=[0, 2])
        wt_m_data = wt_data.loc[wt_data['Metabolite'] == m]
        wt_arr = wt_m_data[['Hours', 'Z-score median', 'AUC OD', 'Z-score std']].sort_values(by='AUC OD').to_numpy().T
        wt_arr = avoid_duplicates(wt_arr, idxs=[0, 2])
        # strain_values = [strain.]
        strain_dict[m] = (s_arr, wt_arr)
    
    # Add OD data
    m_data = sample_od_mean.loc[sample_od_mean.Strain==strain, ['Hours', 'OD_mean', 'OD_std']].to_numpy().T
    wt_data = sample_od_mean.loc[(sample_od_mean.Strain=='WT')&(sample_od_mean.Batch==batch), ['Hours', 'OD_mean', 'OD_std']].to_numpy().T
    
    strain_dict['OD'] = (m_data, wt_data)
    new_data_dict[strain] = strain_dict

In [17]:
save_compressed_pickle(new_data_dict, 'visualization/data/new_data_dict_keio_all_strains_with_std')